# 10 — Capstone: model versus deterministic and hybrid designs

**Estimated time:** 55 minutes<br>
**Prerequisites:** 09 — Capstone policy-derived ground truth<br>
**Learner-produced evidence:** a validation mechanics probe, frozen ceiling check, and explicit architecture choice

## Learning objectives

- Distinguish a one-record mechanics probe from comparative evidence.
- Prove that a hybrid renderer cannot alter authoritative decisions.
- Decide which behavior, if any, justifies a language model.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Not every task that contains text should be delegated end to end to a generative model. In policy review, rules can preserve authoritative decisions while a model improves wording or extracts bounded candidates. A hybrid design makes that authority boundary explicit and gives uncertain or failed model output a deterministic fallback. Architecture is part of the evaluation question.

## Key terms in plain language

- **deterministic system:** a system that produces the same result for the same validated input and versioned rules.
- **generative model:** a probabilistic model that produces sequences and can vary or create unsupported content.
- **hybrid architecture:** a system that assigns different responsibilities to rules, models, external systems, and humans.
- **authority boundary:** the explicit line defining which component is allowed to decide or modify each field.
- **renderer:** a component that converts authoritative structured findings into user-facing wording.
- **normalization:** mapping varied model text into a validated canonical form without changing authoritative meaning.
- **fallback:** a safe alternate behavior used when the model is unavailable, invalid, uncertain, or outside scope.
- **fail closed:** defaulting to a safe non-approval or human-review state when required evidence is missing.


## Mental model — how to think about this

Use the rule **rules decide; models explain** whenever truth is computable from governed facts. Think of the model as an untrusted assistant behind a typed interface: it may propose wording or bounded extractions, but validated code owns authoritative fields. Any model output that crosses that boundary is rejected, normalized, or routed to review.

### Running example

Think of the policy engine as a calculator and the optional model as a copywriter. The calculator owns readiness status, check IDs, and severity. The copywriter can explain those immutable findings. If it times out, emits invalid output, or contradicts a finding, a deterministic renderer takes over without changing the decision.

### Questions to ask before continuing

- Which fields are authoritative, and which component has permission to set each one?
- What capability does the model add that deterministic code cannot provide adequately?
- Can invalid, unavailable, or contradictory model output fall back without changing the decision?
- Does the measured quality gain justify latency, cost, nondeterminism, privacy, and monitoring obligations?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Use the least-necessary model role.** Keep computable policy in rules; use a model only for measured language or extraction capabilities that benefit from it.
- **Constrain and validate the interface.** Allow-list fields and values, forbid model mutation of authoritative decisions, and test adversarial or malformed outputs.
- **Provide a deterministic fallback.** A model outage, timeout, parse failure, or low-confidence result should have a documented safe path.
- **Evaluate each layer and the whole system.** Score rule correctness, model contribution, end-to-end contract, fallback rate, latency, and human-escalation quality separately.
- **Keep provenance visible.** Reviewers should know which findings came from rules, model output, external facts, or human decisions.

## Common mistakes and why they fail

- **Putting the LLM in the authoritative path because the input is text.** This adds uncertainty where validated rules may suffice.
- **Letting generated prose change the decision.** A renderer should express canonical findings, not reinterpret them.
- **Having no failure mode.** Retries are not a safety design; define timeout, invalid-output, outage, and uncertainty behavior.
- **Ignoring the model's operational tax.** A small score gain may not justify new latency, cost, access controls, and monitoring.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)
- **Specification:** [JSON Schema Draft 2020-12](https://json-schema.org/draft/2020-12)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Rehearse the deterministic method on validation

Test is still closed. The deterministic method should be exact on
validation for fully specified rules because the same versioned policy
engine generated the labels. This rehearses the evaluator; it is not yet
the frozen ceiling measurement. A model must justify a different
capability—such as bounded wording—rather than probabilistically
duplicating already-computable policy decisions.


In [ ]:
import json

from aai_local_finetuning.capstone import (
    CAPSTONE_SYSTEM_PROMPT,
    CapstonePrediction,
    build_hybrid_review,
    deterministic_capstone_predictions,
    evaluate_capstone_predictions,
    load_capstone_records,
)
from aai_local_finetuning.evaluation import (
    DeterministicInferenceConfig,
    build_local_mlx_inference_config,
    recheck_evaluation_session,
    start_evaluation_session,
)
from aai_local_finetuning.modeling import LocalMLXPredictor
from aai_local_finetuning.offline import verify_flight_manifest
from aai_local_finetuning.settings import PROJECT_ROOT, load_settings

settings = load_settings()
verify_flight_manifest(settings)
source_dir = PROJECT_ROOT / "data" / "processed" / "capstone-readiness-v1"
validation_records = load_capstone_records(source_dir / "validation.jsonl")
deterministic_validation_session = start_evaluation_session()
deterministic_validation_predictions = deterministic_capstone_predictions(
    validation_records
)
deterministic_validation_report = evaluate_capstone_predictions(
    validation_records,
    deterministic_validation_predictions,
    evaluation_session=deterministic_validation_session,
    inference_config=DeterministicInferenceConfig(
        method="deterministic-validation-policy"
    ),
)
recheck_evaluation_session(deterministic_validation_session)
deterministic_validation_report.aggregate.model_dump(mode="json")

## One untouched-model validation probe — demo, not evidence

This bounded **validation** example shows mechanics, not a winner and not
a model-versus-policy comparison. The compact model contract asks for
status and non-pass checks. One output has no useful uncertainty or slice
coverage; a claimed comparison must score locked methods on identical
records and fingerprints.

The model-aware session starts before model construction and inference,
is supplied to the evaluator, and is rechecked after the report. It binds
governed source, installed packages, and the pinned checkpoint files
across that whole path. The `LocalMLXInferenceConfig` separately records
the prompt recipe and complete greedy decoding settings. Together they
form the evidence boundary, and the output budget comes from that persisted
config rather than a second loose number.


In [ ]:
probe_record = validation_records[0]
model_probe_session = start_evaluation_session(settings)
model_probe_inference_config = build_local_mlx_inference_config(
    model_probe_session,
    method="validation-model-probe",
    prompt_recipe="capstone-compact-basic",
    max_tokens=160,
)
predictor = LocalMLXPredictor(settings.model_dir)
generated = predictor.generate(
    [
        {"role": "system", "content": CAPSTONE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                probe_record.manifest,
                separators=(",", ":"),
                sort_keys=True,
            ),
        },
    ],
    max_tokens=model_probe_inference_config.generation.max_tokens,
)
model_prediction = CapstonePrediction(
    example_id=probe_record.example_id,
    raw_text=generated.text,
    latency_ms=generated.latency_ms,
    output_tokens=generated.output_tokens,
    peak_memory_mb=generated.peak_memory_mb,
)
model_probe_report = evaluate_capstone_predictions(
    (probe_record,),
    (model_prediction,),
    evaluation_session=model_probe_session,
    inference_config=model_probe_inference_config,
)
recheck_evaluation_session(model_probe_session)
{
    "output_preview": generated.text[:500],
    "metrics": model_probe_report.aggregate.model_dump(mode="json"),
    "performance": model_probe_report.performance.model_dump(mode="json"),
    "evidence_status": "mechanics demo only; do not rank methods",
}

## The hybrid authority boundary

A renderer receives a frozen check and may return prose only. It has no
channel for changing status, result, severity, rule ID, or remediation ID.
Empty or failing renderers fall back to deterministic policy wording.


In [ ]:
def learner_renderer(check):
    return f"Review note: {check.evidence}"


hybrid = build_hybrid_review(
    probe_record.manifest,
    renderer=learner_renderer,
    renderer_name="notebook_demo",
)
{
    "status_unchanged": (
        hybrid.deterministic_review.status == probe_record.expected_output.status
    ),
    "checks_unchanged": (
        hybrid.deterministic_review.checks == probe_record.expected_output.checks
    ),
    "explanation_preview": hybrid.explanations[0].model_dump(mode="json"),
}

## Prove fallback—and understand what the type boundary cannot prove

A renderer exception cannot weaken the policy decision: deterministic
wording replaces it. The typed boundary also prevents the renderer from
changing status or severity. It does **not** prove that arbitrary prose is
truthful, so generated explanations still need their own groundedness and
human-review evaluation.


In [ ]:
def failing_renderer(_check):
    raise RuntimeError("simulated renderer outage")


fallback_review = build_hybrid_review(
    probe_record.manifest,
    renderer=failing_renderer,
    renderer_name="simulated_failure",
)
{
    "authoritative_review_unchanged": (
        fallback_review.deterministic_review == hybrid.deterministic_review
    ),
    "fallback_is_nonempty": all(
        explanation.text for explanation in fallback_review.explanations
    ),
    "remaining_risk": (
        "A nonempty generated explanation may still be misleading; "
        "evaluate wording separately."
    ),
}

## Optional capstone LoRA smoke

Keep this disabled for Run All. When enabled it writes a notebook-specific
adapter and leaves the canonical capstone change untouched. Its success
evidence binds the expected base model, exact capstone training files,
governed source, interpreter/platform, and exact package set; a smoke
adapter cannot qualify as the canonical change. Falling loss still does
not beat the deterministic ceiling.


In [ ]:
from aai_local_finetuning.training import run_lora

RUN_CAPSTONE_TRAINING = False
if RUN_CAPSTONE_TRAINING:
    verify_flight_manifest(settings)
    capstone_training = run_lora(
        iterations=10,
        config_path=(PROJECT_ROOT / "configs" / "training" / "capstone-lora.yaml"),
        adapter_path=(
            PROJECT_ROOT / "artifacts" / "notebook" / "adapters" / "capstone-smoke"
        ),
        log_name="notebook-capstone-smoke",
    ).model_dump(mode="json")
else:
    capstone_training = {"status": "skipped"}
capstone_training

## Lock methods, then open the capstone test once

All model probing and optional training now precede this boundary. The
deterministic ceiling is measured on the complete test. The expensive
base-model comparison is opt-in, but when enabled it uses every identical
test record so the two reports share a defensible scope. Leaving it off
produces **missing model evidence**, not permission to infer a winner.

One frozen model-aware evaluation session starts before either method
predicts and closes only after scoring. The deterministic report still
truthfully declares no model dependency; the optional model report carries
the verified checkpoint and decoding contract. Session rechecks make those
claims evidence of unchanged bytes—not merely timestamps near a report.


In [ ]:
verify_flight_manifest(settings)
test_records = load_capstone_records(source_dir / "test.jsonl")
frozen_evaluation_session = start_evaluation_session(settings)
deterministic_predictions = deterministic_capstone_predictions(test_records)
deterministic_report = evaluate_capstone_predictions(
    test_records,
    deterministic_predictions,
    evaluation_session=frozen_evaluation_session,
    inference_config=DeterministicInferenceConfig(method="deterministic-policy"),
)

RUN_FROZEN_MODEL_COMPARISON = False
model_frozen_report = None
if RUN_FROZEN_MODEL_COMPARISON:
    frozen_model_inference_config = build_local_mlx_inference_config(
        frozen_evaluation_session,
        method="untouched-model-basic",
        prompt_recipe="capstone-compact-basic",
        max_tokens=160,
    )
    frozen_predictor = LocalMLXPredictor(settings.model_dir)
    model_predictions = []
    for record in test_records:
        generated = frozen_predictor.generate(
            [
                {"role": "system", "content": CAPSTONE_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": json.dumps(
                        record.manifest,
                        separators=(",", ":"),
                        sort_keys=True,
                    ),
                },
            ],
            max_tokens=(frozen_model_inference_config.generation.max_tokens),
        )
        model_predictions.append(
            CapstonePrediction(
                example_id=record.example_id,
                raw_text=generated.text,
                latency_ms=generated.latency_ms,
                output_tokens=generated.output_tokens,
                peak_memory_mb=generated.peak_memory_mb,
            )
        )
    model_frozen_report = evaluate_capstone_predictions(
        test_records,
        tuple(model_predictions),
        evaluation_session=frozen_evaluation_session,
        inference_config=frozen_model_inference_config,
    )

recheck_evaluation_session(frozen_evaluation_session)
current_evaluation_contract_sha256 = frozen_evaluation_session.execution_contract_sha256
report_execution_contracts = {
    deterministic_report.evaluation_execution_contract_sha256,
    *(
        [model_frozen_report.evaluation_execution_contract_sha256]
        if model_frozen_report is not None
        else []
    ),
}
evaluation_lineage_current = report_execution_contracts == {
    current_evaluation_contract_sha256
}
if not evaluation_lineage_current:
    raise RuntimeError(
        "capstone reports do not match the current source/runtime contract"
    )

frozen_comparison = {
    "records": len(test_records),
    "deterministic_policy": (deterministic_report.aggregate.model_dump(mode="json")),
    "untouched_model": (
        model_frozen_report.aggregate.model_dump(mode="json")
        if model_frozen_report is not None
        else "not run; comparative model evidence is absent"
    ),
    "evaluation_execution_contract_sha256": (current_evaluation_contract_sha256),
    "comparison_complete": (
        model_frozen_report is not None and evaluation_lineage_current
    ),
}
frozen_comparison

## Exercise — choose the production shape

Fill in one row per behavior. Success means deterministic checks retain
authority, unavailable facts route outward, and the model is used only
where probabilistic language actually helps.


In [ ]:
architecture_decision = [
    {
        "behavior": "readiness decision",
        "owner": "deterministic policy engine",
        "reason": "exact, auditable rules already define the answer",
        "failure_handling": "fail closed to not-ready or review",
        "evidence": "policy tests and frozen deterministic report",
    },
    {
        "behavior": "external registry fact",
        "owner": "authorized lookup",
        "reason": "the fact is absent from the local manifest",
        "failure_handling": "route to review; never assume false",
        "evidence": "lookup provenance and authorization record",
    },
    {
        "behavior": "residual risk acceptance",
        "owner": "qualified human",
        "reason": "risk appetite is a governed judgment, not a text prediction",
        "failure_handling": "await an explicit recorded decision",
        "evidence": "reviewer identity, rationale, and timestamp",
    },
    {
        "behavior": "remediation wording",
        "owner": "policy text or constrained tiny-model renderer",
        "reason": "wording may vary without changing authority",
        "failure_handling": "deterministic wording fallback",
        "evidence": "groundedness, policy, latency, and fallback tests",
    },
]
verify_flight_manifest(settings)
recheck_evaluation_session(frozen_evaluation_session)
if frozen_evaluation_session.execution_contract_sha256 != (
    frozen_comparison["evaluation_execution_contract_sha256"]
):
    raise RuntimeError("source/runtime changed after the capstone evidence was written")
architecture_decision

**Hint:** ask what must be correct, what must be looked up, what needs a
person, and what merely benefits from flexible wording.


## Checkpoint

The likely design is deterministic validation plus optional constrained
language generation—not a model pretending to know every readiness fact.

**Next:** `11_design_the_next_project.ipynb` turns the remaining dataset
ideas into review plans without fabricating unverified schemas or rights.
